# Задача 2. Прогнозирование стоимости на N лет вперёд

Сравниваются пять моделей прогноза временного изменения стоимости недвижимости:

1. **DummyRegressor (Naive Forecast)** — нулевая временная поправка: прогноз сохраняет базовую стоимость, рассчитанную на последний известный год.
2. **Ridge Regression** — линейная регрессия с L2-регуляризацией.
3. **HuberRegressor** — робастная линейная регрессия, менее чувствительная к выбросам.
4. **ElasticNet** — линейная регрессия с комбинацией L1- и L2-регуляризации.
5. **MLPRegressor** — многослойная нейронная сеть.

Сохраняется исходная схема прогноза: общий фиксированный CatBoost оценивает базовую стоимость на год обучения, а выбранная модель рассчитывает изменение логарифма цены за квадратный метр между годом обучения и целевым годом. CatBoost не участвует в сравнении кандидатов и одинаков для всех пяти вариантов.

Гиперпараметры подбираются по скользящей временной кросс-валидации на исторических периодах до 2025 года. Итоговое сравнение выполняется на backtest 2026 года: обучение на сделках до 2026 года, прогноз сделок 2026 года. Данные 2026 года не используются ни для подбора параметров, ни при обучении моделей backtest.


## 1. Библиотеки, пути и общие параметры

Подключаются существующие функции очистки и подготовки данных проекта. EDA не выполняется. Настройки общего CatBoost базовой цены фиксированы, чтобы сравнение отражало качество пяти моделей временной поправки.


In [1]:
from pathlib import Path
import sys
import time
import warnings
import inspect
from itertools import product

import numpy as np
import pandas as pd
from IPython.display import display, HTML

from sklearn.dummy import DummyRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge, HuberRegressor, ElasticNet
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning

from catboost import CatBoostRegressor

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = Path.cwd()
CSV_NAME = "СДЕЛКИ_Мос_2023-2026.csv"
while not (ROOT / CSV_NAME).exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / CSV_NAME).exists():
    raise FileNotFoundError(f"Не найден файл данных: {CSV_NAME}")

sys.path.insert(0, str(ROOT / "web_app"))
import model as app_model

DATA_PATH = ROOT / CSV_NAME
RANDOM_STATE = 42
FEATURE_COLS = app_model.FEATURE_COLS
CAT_COLS = app_model.CATEGORICAL_COLS
NUM_COLS = app_model.NUMERIC_COLS
BASE_YEAR = 2026
BACKTEST_MIN_DEALS = 1000

# Общая базовая модель CatBoost — фиксирована и не сравнивается с кандидатами.
BASE_CAT_PARAMS = {
    "iterations": 555,
    "depth": 10,
    "learning_rate": 0.056794,
    "l2_leaf_reg": 2.0917992640459695,
    "random_strength": 0.5598053604013576,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "thread_count": 3,
    "allow_writing_files": False,
}

print("Данные:", DATA_PATH)
print("Базовый год backtest:", BASE_YEAR)
print("Сравниваемые модели: DummyRegressor, Ridge, Huber, ElasticNet, MLPRegressor")
print("Признаки:", FEATURE_COLS)


Данные: C:\Users\AnnaA\Desktop\Prognosis_system\СДЕЛКИ_Мос_2023-2026.csv
Базовый год backtest: 2026
Сравниваемые модели: DummyRegressor, Ridge, Huber, ElasticNet, MLPRegressor
Признаки: ['region', 'district', 'street', 'wall_material_code', 'year_build', 'floor', 'area', 'period_year']


## 2. Подготовка данных и временных fold

Используется существующая очистка данных и последовательная фильтрация выбросов. Для каждого fold границы фильтра вычисляются только по train.

В подборе гиперпараметров проверяются исторические прогнозы, где год обучения строго предшествует целевому году, а целевой год меньше 2026. 2026 год остаётся отдельной итоговой проверкой.


In [2]:
df_raw = app_model.read_csv_robust(DATA_PATH)
df_raw.columns = df_raw.columns.astype(str).str.strip().str.lower()
clean_df = app_model.clean_data(df_raw)


def prepare_backtest_pair(train_raw, test_raw):
    """Применяет фильтры проекта; границы выбросов оцениваются только на train."""
    train = train_raw.copy()
    test = test_raw.copy()
    train["price_per_sqm"] = train["price"] / train["area"]
    test["price_per_sqm"] = test["price"] / test["area"]

    for column in ("price", "price_per_sqm"):
        mean = float(train[column].mean())
        std = float(train[column].std())
        if not np.isfinite(std) or std <= 0:
            continue
        lower, upper = mean - 3 * std, mean + 3 * std
        train = train[train[column].between(lower, upper)].copy()
        test = test[test[column].between(lower, upper)].copy()

    def finalize(frame):
        frame = frame.copy()
        for column in CAT_COLS:
            frame[column] = frame[column].map(app_model.normalize_text)
        frame["wall_material_code"] = (
            frame["wall_material_code"].astype(str).str.replace(r"\.0+$", "", regex=True)
        )
        for column in list(NUM_COLS) + ["price", "period_year"]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        frame = frame.dropna(subset=list(NUM_COLS) + ["price", "period_year"]).copy()
        frame = frame[
            frame["period_year"].between(2023, 2026)
            & frame["year_build"].between(1800, 2027)
            & (frame["year_build"] <= frame["period_year"] + 1)
            & frame["floor"].between(1, 100)
            & frame["area"].between(25, 400)
            & frame["price"].between(7_000_000, 1_000_000_000)
        ].copy()
        frame["price_per_sqm"] = frame["price"] / frame["area"]
        frame = frame[frame["price_per_sqm"].between(50_000, 15_000_000)].copy()
        frame["period_year"] = frame["period_year"].astype(int)
        return frame.sort_values("period_start_date").reset_index(drop=True)

    return finalize(train), finalize(test)


available_years = sorted(clean_df["period_year"].dropna().astype(int).unique())
cv_folds = []

for origin_year in available_years:
    if origin_year >= BASE_YEAR:
        continue
    for target_year in available_years:
        if origin_year < target_year < BASE_YEAR:
            raw_train = clean_df[clean_df["period_year"] <= origin_year].copy()
            raw_test = clean_df[clean_df["period_year"] == target_year].copy()
            if len(raw_train) < 30 or len(raw_test) < BACKTEST_MIN_DEALS:
                continue

            train_fold, test_fold = prepare_backtest_pair(raw_train, raw_test)
            if len(train_fold) >= 30 and len(test_fold) >= BACKTEST_MIN_DEALS:
                cv_folds.append((origin_year, target_year, train_fold, test_fold))

if not cv_folds:
    raise ValueError("Для временной кросс-валидации недостаточно исторических fold.")

print("Fold для подбора:", [(origin, target, len(test)) for origin, target, _, test in cv_folds])
print("Данные 2026 года будут использованы только в итоговом backtest.")


Fold для подбора: [(np.int64(2023), np.int64(2024), 14682), (np.int64(2023), np.int64(2025), 19305), (np.int64(2024), np.int64(2025), 23747)]
Данные 2026 года будут использованы только в итоговом backtest.


## 3. Сравниваемые модели прогноза

Все регрессионные кандидаты обучаются на `log(price / area)` и получают одинаковое представление признаков. Временной признак включён для Ridge, Huber, ElasticNet и MLPRegressor. Числовые и категориальные признаки кодируются через `DictVectorizer`, затем масштабируются без центрирования, чтобы сохранить разреженную матрицу.

**DummyRegressor** задаёт нулевое изменение временной поправки. Поэтому прогноз остаётся равен базовой оценке стоимости на год обучения — это наивный сценарий «цена не изменилась». У такой модели нет содержательных настраиваемых параметров; `strategy="constant", constant=0.0` фиксирует именно этот сценарий.

Для HuberRegressor разреженная матрица преобразуется в плотную, поскольку эта модель работает с плотным массивом признаков. Остальные кандидаты используют разреженную матрицу.


In [3]:
MODEL_DUMMY = "DummyRegressor (Naive Forecast)"
MODEL_RIDGE = "Ridge Regression"
MODEL_HUBER = "HuberRegressor"
MODEL_ELASTIC = "ElasticNet"
MODEL_MLP = "MLPRegressor"


def catboost_frame(frame):
    result = frame[FEATURE_COLS].copy()
    for column in CAT_COLS:
        result[column] = result[column].astype(str)
    return result


def fit_base_catboost(train):
    """Общая для всех вариантов CatBoost-модель базовой цены."""
    model = CatBoostRegressor(**BASE_CAT_PARAMS)
    X = catboost_frame(train)
    y = np.log(train["price"].to_numpy(float) / train["area"].to_numpy(float))
    model.fit(X, y, cat_features=CAT_COLS)
    return model


def predict_base_log(model, X):
    return model.predict(catboost_frame(X))


def make_year_weights(train):
    """Выравнивает суммарный вес сделок по годам обучения."""
    year_counts = train["period_year"].value_counts()
    weights = train["period_year"].map(
        lambda year: len(train) / (len(year_counts) * year_counts.loc[year])
    ).to_numpy(float)
    return weights


def make_candidate_matrix(frame, support, vectorizer=None, scaler=None, fit=False):
    records = [
        app_model.record_features(
            row,
            support,
            with_time=True,
            trend_scale=1.0,
        )
        for row in frame[FEATURE_COLS].to_dict("records")
    ]

    if fit:
        vectorizer = DictVectorizer(sparse=True)
        matrix = vectorizer.fit_transform(records)
        scaler = StandardScaler(with_mean=False)
        matrix = scaler.fit_transform(matrix)
        return matrix, vectorizer, scaler

    matrix = vectorizer.transform(records)
    return scaler.transform(matrix)


def fit_with_optional_sample_weight(estimator, X, y, sample_weight):
    """Передаёт веса, если установленная версия sklearn поддерживает sample_weight."""
    fit_parameters = inspect.signature(estimator.fit).parameters
    if "sample_weight" in fit_parameters:
        estimator.fit(X, y, sample_weight=sample_weight)
    else:
        estimator.fit(X, y)
    return estimator


def fit_candidate(name, params, train):
    """Обучает кандидата на log(price / area) либо на нулевую временную поправку."""
    fitted = {"name": name, "params": params.copy()}

    # Нулевая поправка означает сохранение базовой стоимости без роста/снижения.
    if name == MODEL_DUMMY:
        estimator = DummyRegressor(
            strategy=params["strategy"],
            constant=params["constant"],
        )
        dummy_X = np.zeros((len(train), 1), dtype=float)
        dummy_y = np.zeros(len(train), dtype=float)
        estimator.fit(dummy_X, dummy_y)
        fitted.update(model=estimator, no_change=True)
        return fitted

    y = np.log(train["price"].to_numpy(float) / train["area"].to_numpy(float))
    sample_weight = make_year_weights(train)
    support = app_model.make_support(train)
    matrix, vectorizer, scaler = make_candidate_matrix(
        train, support, fit=True
    )

    if name == MODEL_RIDGE:
        estimator = Ridge(
            alpha=params["alpha"],
            solver="lsqr",
            fit_intercept=True,
            tol=1e-4,
            max_iter=5000,
        )
        X_model = matrix

    elif name == MODEL_HUBER:
        estimator = HuberRegressor(
            epsilon=params["epsilon"],
            alpha=params["alpha"],
            fit_intercept=True,
            max_iter=params["max_iter"],
            tol=1e-5,
        )
        X_model = matrix.toarray()

    elif name == MODEL_ELASTIC:
        estimator = ElasticNet(
            alpha=params["alpha"],
            l1_ratio=params["l1_ratio"],
            fit_intercept=True,
            max_iter=10000,
            tol=1e-4,
            selection="cyclic",
            random_state=RANDOM_STATE,
        )
        X_model = matrix

    elif name == MODEL_MLP:
        estimator = MLPRegressor(
            hidden_layer_sizes=params["hidden_layer_sizes"],
            activation=params["activation"],
            alpha=params["alpha"],
            learning_rate_init=params["learning_rate_init"],
            solver="adam",
            max_iter=params["max_iter"],
            early_stopping=False,
            n_iter_no_change=30,
            tol=1e-4,
            random_state=RANDOM_STATE,
        )
        X_model = matrix

    else:
        raise ValueError(f"Неизвестный алгоритм: {name}")

    fit_with_optional_sample_weight(estimator, X_model, y, sample_weight)

    fitted.update(
        model=estimator,
        support=support,
        vectorizer=vectorizer,
        scaler=scaler,
        dense=(name == MODEL_HUBER),
    )
    return fitted


def predict_candidate(fitted, X):
    name = fitted["name"]

    if name == MODEL_DUMMY:
        dummy_X = np.zeros((len(X), 1), dtype=float)
        return fitted["model"].predict(dummy_X)

    matrix = make_candidate_matrix(
        X,
        fitted["support"],
        vectorizer=fitted["vectorizer"],
        scaler=fitted["scaler"],
        fit=False,
    )
    if fitted["dense"]:
        matrix = matrix.toarray()
    return fitted["model"].predict(matrix)


def forecast_log_price(base_model, trend_model, test):
    """Базовая цена × временная поправка выбранной модели."""
    origin_year = int(trend_model["origin_year"])
    target_year = int(trend_model["target_year"])

    X_origin = test[FEATURE_COLS].copy()
    X_target = test[FEATURE_COLS].copy()
    X_origin["period_year"] = origin_year
    X_target["period_year"] = target_year

    base_log = predict_base_log(base_model, X_origin)
    trend_origin = predict_candidate(trend_model, X_origin)
    trend_target = predict_candidate(trend_model, X_target)

    return base_log + trend_target - trend_origin


# Сетка подбора параметров. Для DummyRegressor фиксируется единственный
# осмысленный вариант: нулевая временная поправка.
PARAMETER_GRIDS = {
    MODEL_DUMMY: [
        {"strategy": "constant", "constant": 0.0},
    ],
    MODEL_RIDGE: [
        {"alpha": alpha}
        for alpha in [0.1, 1.0, 10.0, 100.0, 500.0]
    ],
    MODEL_HUBER: [
        {"epsilon": epsilon, "alpha": alpha, "max_iter": 1000}
        for epsilon, alpha in product(
            [1.35, 1.5],
            [0.0001, 0.001, 0.01],
        )
    ],
    MODEL_ELASTIC: [
        {"alpha": alpha, "l1_ratio": l1_ratio}
        for alpha, l1_ratio in product(
            [0.001, 0.01, 0.1],
            [0.2, 0.5, 0.8],
        )
    ],
    MODEL_MLP: [
        {
            "hidden_layer_sizes": hidden_layer_sizes,
            "activation": "relu",
            "alpha": alpha,
            "learning_rate_init": 0.001,
            "max_iter": 300,
        }
        for hidden_layer_sizes, alpha in product(
            [(32,), (64,), (64, 32)],
            [0.0001, 0.001],
        )
    ],
}


## 4. Подбор гиперпараметров временной кросс-валидацией

Для каждого fold общий CatBoost базовой цены обучается один раз и используется всеми кандидатами. Каждая модель оценивается по MAE логарифма итоговой прогнозной цены за квадратный метр после применения временной поправки.

Для каждого алгоритма выбирается набор параметров с минимальной средней `CV MAE (log)`. Для DummyRegressor используется единственный фиксированный вариант наивного прогноза: нулевое изменение цены.

Выводимые здесь строки с `CV MAE(log)`, лучшими параметрами и временем подбора являются диагностикой процесса оптимизации, а не итоговой сравнительной таблицей. Итоговая таблица формируется в следующей секции.


In [4]:
# Кэшируем общий CatBoost для каждого CV-fold.
if "base_models_by_fold" not in globals():
    base_models_by_fold = {}
    for origin_year, target_year, train_fold, _ in cv_folds:
        base_models_by_fold[(origin_year, target_year)] = fit_base_catboost(train_fold)

# Сохраняем результаты завершённых конфигураций, чтобы можно было продолжить
# подбор после прерывания выполнения ячейки.
if "cv_scores" not in globals() or set(cv_scores) != set(PARAMETER_GRIDS):
    cv_scores = {name: [] for name in PARAMETER_GRIDS}
if "best_params" not in globals() or set(best_params) != set(PARAMETER_GRIDS):
    best_params = {}
if "tuning_times" not in globals() or set(tuning_times) != set(PARAMETER_GRIDS):
    tuning_times = {}

for name, grid in PARAMETER_GRIDS.items():
    scores = cv_scores.setdefault(name, [])
    if len(scores) >= len(grid) and name in best_params:
        print(f"{name}: подбор уже завершён; параметры={best_params[name]}")
        continue

    started = time.perf_counter()

    for params in grid[len(scores):]:
        fold_errors = []

        for origin_year, target_year, train_fold, test_fold in cv_folds:
            base_model = base_models_by_fold[(origin_year, target_year)]
            candidate = fit_candidate(name, params, train_fold)
            candidate["origin_year"] = origin_year
            candidate["target_year"] = target_year

            predicted_log_ppsm = forecast_log_price(
                base_model, candidate, test_fold
            )
            actual_log_ppsm = np.log(
                test_fold["price"].to_numpy(float)
                / test_fold["area"].to_numpy(float)
            )
            fold_errors.extend(
                np.abs(actual_log_ppsm - predicted_log_ppsm).tolist()
            )

        cv_mae_log = float(np.mean(fold_errors))
        scores.append((cv_mae_log, params.copy()))

    best_score, best_config = min(scores, key=lambda item: item[0])
    best_params[name] = best_config.copy()
    tuning_times[name] = tuning_times.get(name, 0.0) + (
        time.perf_counter() - started
    )

    print(
        f"{name}: CV MAE(log)={best_score:.6f}; "
        f"лучшие параметры={best_config}; "
        f"время подбора={tuning_times[name]:.1f} с"
    )

cv_best_log_mae = {
    name: min(score for score, _ in cv_scores[name])
    for name in PARAMETER_GRIDS
}

print("\nПодбор гиперпараметров завершён.")


DummyRegressor (Naive Forecast): CV MAE(log)=0.214118; лучшие параметры={'strategy': 'constant', 'constant': 0.0}; время подбора=2.2 с
Ridge Regression: CV MAE(log)=0.228942; лучшие параметры={'alpha': 500.0}; время подбора=88.6 с
HuberRegressor: CV MAE(log)=4.100429; лучшие параметры={'epsilon': 1.35, 'alpha': 0.01, 'max_iter': 1000}; время подбора=2667.9 с
ElasticNet: CV MAE(log)=0.209718; лучшие параметры={'alpha': 0.1, 'l1_ratio': 0.5}; время подбора=148.1 с
MLPRegressor: CV MAE(log)=1.590773; лучшие параметры={'hidden_layer_sizes': (64, 32), 'activation': 'relu', 'alpha': 0.001, 'learning_rate_init': 0.001, 'max_iter': 300}; время подбора=2217.7 с

Подбор гиперпараметров завершён.


## 5. Итоговый backtest 2026 года

После подбора каждая модель один раз заново обучается на сделках до 2026 года включительно и прогнозирует сделки 2026 года. Общий CatBoost базовой цены одинаков для всех пяти алгоритмов.

В итоговую таблицу выводятся метрики в заданном порядке. `Time, s` — только время одного финального обучения соответствующей модели на её лучших гиперпараметрах (включая подготовку признаков для этой модели). Время перебора гиперпараметров и время обучения общего CatBoost в этот столбец не входят.

**Важно:** ячейка 4 выполняет подбор гиперпараметров и печатает диагностические строки по моделям. Таблица появляется после выполнения следующей кодовой ячейки — раздел 5.


In [5]:
train_raw = clean_df[clean_df["period_year"] < BASE_YEAR].copy()
test_raw = clean_df[clean_df["period_year"] == BASE_YEAR].copy()

train_2026, test_2026 = prepare_backtest_pair(train_raw, test_raw)
if len(train_2026) < 30 or len(test_2026) < BACKTEST_MIN_DEALS:
    raise ValueError(
        "Для backtest 2026 года недостаточно подходящих обучающих или тестовых сделок."
    )

base_model_2026 = fit_base_catboost(train_2026)
result_rows = []

for name in PARAMETER_GRIDS:
    started = time.perf_counter()

    fit_started = time.perf_counter()
    candidate = fit_candidate(name, best_params[name], train_2026)
    model_fit_time = time.perf_counter() - fit_started
    candidate["origin_year"] = BASE_YEAR - 1
    candidate["target_year"] = BASE_YEAR

    predicted_log_ppsm = forecast_log_price(
        base_model_2026, candidate, test_2026
    )
    predicted_price = (
        np.exp(np.clip(predicted_log_ppsm, -20, 25))
        * test_2026["area"].to_numpy(float)
    )
    actual_price = test_2026["price"].to_numpy(float)

    mae = mean_absolute_error(actual_price, predicted_price) / 1e6
    rmse = np.sqrt(mean_squared_error(actual_price, predicted_price)) / 1e6
    mape = (
        np.mean(
            np.abs(actual_price - predicted_price)
            / np.maximum(actual_price, 1.0)
        )
        * 100
    )
    r2 = r2_score(actual_price, predicted_price)

    # Только время одного обучения модели на лучших параметрах.
    # Подбор гиперпараметров, CatBoost и прогнозирование не учитываются.

    result_rows.append({
        "Algorithm": name,
        "MAE,<br><br>million RUB": mae,
        "RMSE,<br><br>million RUB": rmse,
        "MAPE,<br><br>%": mape,
        "R²": r2,
        "CV<br><br>MAE (log)": cv_best_log_mae[name],
        "Time,<br><br>s": model_fit_time,
        "Accuracy, %": 100.0 - mape,
    })

TABLE_COLUMNS = [
    "Algorithm",
    "MAE,<br><br>million RUB",
    "RMSE,<br><br>million RUB",
    "MAPE,<br><br>%",
    "R²",
    "CV<br><br>MAE (log)",
    "Time,<br><br>s",
    "Accuracy, %",
]

comparison_results = (
    pd.DataFrame(result_rows)
    .sort_values("MAE,<br><br>million RUB", ascending=True)
    .reset_index(drop=True)
)

display(HTML(
    comparison_results[TABLE_COLUMNS].to_html(
        index=False,
        escape=False,
        float_format=lambda value: f"{value:.4f}",
    )
))

print(f"Обучающих сделок до 2025 года включительно: {len(train_2026):,}")
print(f"Тестовых сделок 2026 года: {len(test_2026):,}")


Algorithm,"MAE,million RUB","RMSE,million RUB","MAPE,%",R²,CVMAE (log),"Time,s","Accuracy, %"
DummyRegressor (Naive Forecast),5.9871,9.5149,21.0433,0.5425,0.2141,0.0018,78.9567
ElasticNet,6.1058,9.4910,21.1791,0.5448,0.2097,3.9612,78.8209
Ridge Regression,6.6572,9.8665,24.2865,0.5080,0.2289,4.1654,75.7135
HuberRegressor,6.8756,10.1821,25.4473,0.4761,4.1004,260.6034,74.5527
MLPRegressor,15.7614,32.4565,58.7026,-4.3238,1.5908,1421.9866,41.2974


Обучающих сделок до 2025 года включительно: 45,017
Тестовых сделок 2026 года: 15,489


## 6. Интерпретация результата

Основной критерий выбора победителя — MAE на отложенном backtest 2026 года: чем ниже значение, тем меньше средняя абсолютная ошибка прогноза стоимости.

- `RMSE` сильнее штрафует крупные ошибки.
- `MAPE` показывает среднюю относительную ошибку в процентах.
- `R²` отражает долю объяснённой вариации целевой стоимости.
- `CV MAE (log)` показывает устойчивость на исторических временных fold и использовалась для подбора гиперпараметров.
- `Accuracy, %` в таблице рассчитывается как `100 − MAPE`. Это условный показатель, а не отдельная стандартная метрика регрессии.

Последняя ячейка автоматически выводит победившую по MAE модель, лучшие параметры каждой модели и параметры победителя.


In [6]:
# Победитель определяется по минимальной MAE на backtest 2026 года.
winner_name = comparison_results.iloc[0]["Algorithm"]
winner_params = best_params[winner_name]

print("=" * 72)
print("ПОБЕДИТЕЛЬ СРАВНЕНИЯ")
print("=" * 72)
print(f"Модель: {winner_name}")
print(
    "MAE на backtest 2026: "
    f"{comparison_results.iloc[0]['MAE,<br><br>million RUB']:.4f} млн руб."
)
print(f"Лучшие гиперпараметры победителя: {winner_params}")

print("\n" + "=" * 72)
print("ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ ВСЕХ МОДЕЛЕЙ")
print("=" * 72)

for model_name in PARAMETER_GRIDS:
    print(f"\n{model_name}:")
    for parameter, value in best_params[model_name].items():
        print(f"  {parameter}: {value}")


ПОБЕДИТЕЛЬ СРАВНЕНИЯ
Модель: DummyRegressor (Naive Forecast)
MAE на backtest 2026: 5.9871 млн руб.
Лучшие гиперпараметры победителя: {'strategy': 'constant', 'constant': 0.0}

ЛУЧШИЕ ГИПЕРПАРАМЕТРЫ ВСЕХ МОДЕЛЕЙ

DummyRegressor (Naive Forecast):
  strategy: constant
  constant: 0.0

Ridge Regression:
  alpha: 500.0

HuberRegressor:
  epsilon: 1.35
  alpha: 0.01
  max_iter: 1000

ElasticNet:
  alpha: 0.1
  l1_ratio: 0.5

MLPRegressor:
  hidden_layer_sizes: (64, 32)
  activation: relu
  alpha: 0.001
  learning_rate_init: 0.001
  max_iter: 300
